# Data Pipeline — MusicProject

Runs the full preprocessing pipeline on Google Colab.

## Two execution paths

| | Path A — Colab (recommended) | Path B — Local → upload |
|---|---|---|
| Where processing runs | This notebook | Your local machine |
| `out_root` | Drive path | Local temp dir |
| Data lands on Drive? | Directly (Drive IS out_root) | After `drive_sync.upload_song_to_drive()` |
| Local machine cleaned? | N/A | After `drive_sync.clean_local_song()` |

Run **Section 1** (setup) regardless of path.
Then run **Section 2** (Path A) **or** **Section 3** (Path B instructions).
Finish with **Section 4** (split dataset).

---

### §37 cell-header convention

Every code cell in this notebook is preceded by a markdown header that
follows the project standard (ENGINEERING_DECISIONS §37, see
[colab/README.md](README.md)):

> **What this does.** plain-English summary.  
> **Inputs.** files / variables / env read.  
> **Outputs.** files / variables / state written.  
> **Action required.** anything the user must edit, click, approve, or run
> elsewhere. Prefix the heading with **⚠** when the action requires switching
> machines (e.g. F1 backfill needs `basic_pitch_env` locally).  
> **Runtime.** order-of-magnitude (seconds / minutes / hours).

Stub headers below carry `TODO` placeholders — fill them in when you next
edit the cell. `colab/postprocessing.ipynb` is the reference implementation.


---
## Section 1 — Setup (run always)

<!-- §37-stub -->
## Cell 1 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
# 1a. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT      = '/content/drive/MyDrive/MusicProject'
DATA_ROOT       = f'{DRIVE_ROOT}/MusicProjectData'
MANIFEST_DIR    = f'{DRIVE_ROOT}/data'
CHECKPOINT_DIR  = f'{DRIVE_ROOT}/checkpoints'
LOG_DIR         = f'{DRIVE_ROOT}/logs'

print('Drive mounted.')
print(f'DRIVE_ROOT = {DRIVE_ROOT}')

<!-- §37-stub -->
## Cell 2 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
# 1b. Clone repo and install dependencies
import os

REPO_URL   = 'https://github.com/galgeva1/MusicProject.git'
REPO_LOCAL = '/content/MusicProject'

if not os.path.exists(REPO_LOCAL):
    !git clone {REPO_URL} {REPO_LOCAL}
else:
    !git -C {REPO_LOCAL} pull

%cd {REPO_LOCAL}
!pip install -q -r requirements_ml.txt
!pip install -q basic-pitch
print('Dependencies installed.')


<!-- §37-stub -->
## Cell 3 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
# 1c. Create canonical Drive layout (idempotent)
from pathlib import Path

_dirs = [
    DATA_ROOT,
    MANIFEST_DIR,
    CHECKPOINT_DIR,
    f'{CHECKPOINT_DIR}/slakh_sanity',
    f'{CHECKPOINT_DIR}/israeli_v0',
    LOG_DIR,
    f'{LOG_DIR}/slakh_sanity',
    f'{LOG_DIR}/israeli_v0',
    f'{DRIVE_ROOT}/slakh_raw',
    f'{DRIVE_ROOT}/slakh_processed',
]
for d in _dirs:
    Path(d).mkdir(parents=True, exist_ok=True)

print('Drive layout ready:')
!ls {DRIVE_ROOT}

---
## Section 2 — Path A: Batch ingest directly on Colab

Requires `batch_songs.csv` to already be on Drive at `DRIVE_ROOT/batch_songs.csv`.

Processing writes segment tensors directly into Drive — nothing lands on Colab's ephemeral disk.

<!-- §37-stub -->
## Cell 4 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
# 2a. Preview batch_songs.csv
import pandas as pd

batch_csv = f'{DRIVE_ROOT}/batch_songs.csv'
df = pd.read_csv(batch_csv)
print(f'{len(df)} total rows,  {df["enabled"].eq("true").sum()} enabled')
df.head(10)

<!-- §37-stub -->
## Cell 5 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
# 2b. Run batch ingest
# Each song's tensors are written directly to Drive (out_root = DATA_ROOT).
# Resume-safe: already-processed songs are skipped automatically.

!python preprocessing/batch_ingest.py \
    --csv            "{DRIVE_ROOT}/batch_songs.csv" \
    --out_root       "{DATA_ROOT}" \
    --manifest_out   "{MANIFEST_DIR}/dataset_manifest.csv" \
    --resolved_csv   "{DRIVE_ROOT}/batch_songs_resolved.csv" \
    --log            "{DRIVE_ROOT}/batch_ingest_log.csv"

<!-- §37-stub -->
## Cell 6 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
# 2c. Quick sanity check — inspect the consolidated manifest
import pandas as pd

manifest = pd.read_csv(f'{MANIFEST_DIR}/dataset_manifest.csv')
total_hours = manifest['duration_s'].sum() / 3600
print(f'Total segments : {len(manifest)}')
print(f'Total duration : {total_hours:.2f} h')
print(f'Songs          : {manifest["song_id"].nunique()}')
manifest.head()

---
## Section 3 — Path B: Local preprocessing → upload → clean

Run these commands **on your local machine** (not here), then come back to Section 4.

```bash
# 1. Process a single song locally
python process_song_offline.py \
    --url "https://youtu.be/..." \
    --out_root C:/tmp/MusicProjectData

# 2. Upload to Drive and clean local copy
#    (fill in your Drive MusicProjectData folder ID)
python - <<'EOF'
from pathlib import Path
from preprocessing.drive_sync import upload_song_to_drive, clean_local_song

DRIVE_MUSIC_DATA_ID = "<your Drive folder ID>"   # ← fill in once
song_dir = Path("C:/tmp/MusicProjectData/Artist/Album/Song")

upload_song_to_drive(song_dir, DRIVE_MUSIC_DATA_ID)
clean_local_song(song_dir)
EOF

# 3. To batch-process all songs locally:
python preprocessing/batch_ingest.py \
    --csv            batch_songs.csv \
    --out_root       C:/tmp/MusicProjectData \
    --manifest_out   C:/tmp/MusicProjectData/data/dataset_manifest.csv \
    --resolved_csv   batch_songs_resolved.csv \
    --log            batch_ingest_log.csv \
    --upload_to_drive <your Drive folder ID> \
    --clean_after_upload
```

After all songs are on Drive, continue with Section 4 here in Colab.

---
## Section 4 — Split dataset into train / val / test

<!-- §37-stub -->
## Cell 7 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
# 4a. Run split_dataset.py
# Groups by song_id so no song leaks across splits.

!python preprocessing/split_dataset.py \
    --manifest  "{MANIFEST_DIR}/dataset_manifest.csv" \
    --out_dir   "{MANIFEST_DIR}" \
    --train 0.8 --val 0.1 --test 0.1 \
    --seed 42

<!-- §37-stub -->
## Cell 8 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
# 4b. Verify splits
import pandas as pd

for split in ['train', 'val', 'test']:
    df = pd.read_csv(f'{MANIFEST_DIR}/{split}.csv')
    hours = df['duration_s'].sum() / 3600
    print(f'{split:5s}  songs={df["song_id"].nunique():3d}  '
          f'segments={len(df):5d}  {hours:.2f} h')

# Leakage check
train_ids = set(pd.read_csv(f'{MANIFEST_DIR}/train.csv')['song_id'])
val_ids   = set(pd.read_csv(f'{MANIFEST_DIR}/val.csv')  ['song_id'])
test_ids  = set(pd.read_csv(f'{MANIFEST_DIR}/test.csv') ['song_id'])
assert train_ids.isdisjoint(val_ids),  'LEAK: train ∩ val'
assert train_ids.isdisjoint(test_ids), 'LEAK: train ∩ test'
assert val_ids.isdisjoint(test_ids),   'LEAK: val ∩ test'
print('\n✓ No song-level leakage across splits')

### NEW_CELL
---
## Section 5 — Slakh2100 download + filter (STEP 3 of REMAINING_WORKPLAN)

Pulls `slakh2100_flac_redux.tar.gz` (≈ 145 GB) from Zenodo record 4599666 and
streams it directly into per-track folders under `$DRIVE_ROOT/slakh_raw/`,
keeping only tracks whose stems include both **bass** and **drums** (rough
proxy for rock/pop instrumentation, since the Zenodo metadata.yaml does not
include a genre tag).

**Cap:** `MAX_TRACKS = 60`. **Target total duration:** 2.5 – 3.5 h.

> **Disk note:** uses a streaming tarfile reader so the full archive is
> never written to Colab disk; only ~50 MB at a time is buffered in memory.
> Download itself can take several hours on Colab's bandwidth — run on a
> Colab Pro+ runtime to keep it alive, or restart with `--continue` if needed.


<!-- §37-stub -->
## Cell 9 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
### NEW_CELL
# 5a. Stream-download Slakh2100 from Zenodo and filter to rock/pop-like tracks
#     (= tracks containing both Bass and Drums stems).
#
# Output: SLAKH_RAW/Track*/  (each containing mix.flac + MIDI/*.mid + metadata.yaml)

import io
import os
import re
import shutil
import sys
import tarfile
import yaml
from pathlib import Path

# Make sure pyyaml + requests are present
!pip install -q pyyaml requests
import requests

ZENODO_URL  = 'https://zenodo.org/records/4599666/files/slakh2100_flac_redux.tar.gz'
SLAKH_RAW_P = Path(SLAKH_RAW)
SLAKH_RAW_P.mkdir(parents=True, exist_ok=True)

MAX_TRACKS  = 60
BASS_HINTS  = ('bass',)
DRUMS_HINTS = ('drum', 'percussion')

# Skip if we already have enough tracks (idempotent re-runs)
existing = sorted(p for p in SLAKH_RAW_P.iterdir() if p.is_dir() and p.name.startswith('Track'))
print(f'Existing tracks in {SLAKH_RAW_P}: {len(existing)}')
if len(existing) >= MAX_TRACKS:
    print(f'Already have {len(existing)} >= MAX_TRACKS={MAX_TRACKS} — skipping download.')
else:
    needed = MAX_TRACKS - len(existing)
    already = {p.name for p in existing}
    print(f'Need {needed} more tracks. Streaming Slakh tar...')

    selected = 0
    cur_track = None         # current Track000NN
    cur_buf   = {}           # member_name -> bytes (only for cur_track)

    def stem_classes(meta: dict):
        stems = meta.get('stems') or {}
        return [str(v.get('inst_class', '')).lower() for v in stems.values()]

    def keep_track(meta: dict) -> bool:
        classes = stem_classes(meta)
        if not classes:
            return False
        has_bass  = any(any(h in c for h in BASS_HINTS)  for c in classes)
        has_drums = any(any(h in c for h in DRUMS_HINTS) for c in classes)
        return has_bass and has_drums

    def flush_track(track_name: str, members: dict):
        global selected
        # Locate metadata.yaml in members
        meta_key = next((k for k in members if k.endswith('metadata.yaml')), None)
        if meta_key is None:
            return  # no metadata, skip silently
        try:
            meta = yaml.safe_load(members[meta_key])
        except Exception as e:
            print(f'  {track_name}: metadata parse error ({e}) — skipping')
            return
        if not keep_track(meta):
            return
        # Write accepted track to Drive
        track_dir = SLAKH_RAW_P / track_name
        midi_dir  = track_dir / 'MIDI'
        track_dir.mkdir(parents=True, exist_ok=True)
        midi_dir.mkdir(exist_ok=True)
        for name, blob in members.items():
            base = Path(name).name
            if name.endswith('metadata.yaml'):
                (track_dir / 'metadata.yaml').write_bytes(blob)
            elif name.endswith('mix.flac'):
                (track_dir / 'mix.flac').write_bytes(blob)
            elif '/MIDI/' in name and (name.endswith('.mid') or name.endswith('.midi')):
                (midi_dir / base).write_bytes(blob)
        selected += 1
        n_classes = len(stem_classes(meta))
        print(f'  [{selected:3d}/{MAX_TRACKS}] kept {track_name}  '
              f'({n_classes} stems, {len(members)} files)')

    track_re = re.compile(r'(Track\d{5})')

    with requests.get(ZENODO_URL, stream=True, timeout=60) as r:
        r.raise_for_status()
        with tarfile.open(fileobj=r.raw, mode='r|gz') as tar:
            for member in tar:
                if not member.isfile():
                    continue
                m = track_re.search(member.name)
                if not m:
                    continue
                track_name = m.group(1)

                # If we moved to a new track, finalize the previous one
                if cur_track is not None and track_name != cur_track:
                    if cur_track not in already:
                        flush_track(cur_track, cur_buf)
                    cur_buf = {}
                    if selected >= needed:
                        print('Reached MAX_TRACKS — stopping stream.')
                        break

                cur_track = track_name
                # Only buffer files we might need (saves memory)
                base = Path(member.name).name
                if base == 'metadata.yaml' or base == 'mix.flac' \
                        or member.name.endswith('.mid') or member.name.endswith('.midi'):
                    f = tar.extractfile(member)
                    if f is not None:
                        cur_buf[member.name] = f.read()

            # Final flush
            if cur_track and cur_track not in already and selected < needed:
                flush_track(cur_track, cur_buf)

    print(f'Done. Newly selected tracks: {selected}')

# Re-list to confirm
final = sorted(p for p in SLAKH_RAW_P.iterdir() if p.is_dir() and p.name.startswith('Track'))
print(f'Total tracks now in {SLAKH_RAW_P}: {len(final)}')


<!-- §37-stub -->
## Cell 10 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
### NEW_CELL
# 5b. Acceptance check: total selected audio duration in [2.5, 3.5] h
import soundfile as sf
from pathlib import Path

total_s = 0.0
n_ok    = 0
for td in sorted(Path(SLAKH_RAW).iterdir()):
    mix = td / 'mix.flac'
    if not mix.exists():
        continue
    try:
        info = sf.info(str(mix))
        total_s += info.duration
        n_ok += 1
    except Exception as e:
        print(f'  WARN: could not read {mix}: {e}')

hours = total_s / 3600
print(f'Selected tracks  : {n_ok}')
print(f'Total duration   : {hours:.2f} h')
LO, HI = 2.5, 3.5
if LO <= hours <= HI:
    print(f'STEP 3 ACCEPTANCE PASSED:  duration {hours:.2f} h ∈ [{LO}, {HI}]')
else:
    print(f'STEP 3 ACCEPTANCE NOT MET: duration {hours:.2f} h not in [{LO}, {HI}]')
    print('   → Adjust MAX_TRACKS in cell 5a, or relax/refine the bass+drums filter.')
